In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

# Function to construct an explicit all-string schema

def create_string_schema(columns):

    return StructType([
        StructField(column, StringType(), True)
        for column in columns
    ])

In [0]:
orders_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_schema = create_string_schema(orders_columns)

In [0]:
RAW_PATH = "/Volumes/olist_mlops/bronze/raw_files/"

orders_path = (
    RAW_PATH + "olist_orders_dataset.csv"
)

print(orders_path)

In [0]:
orders_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("enforceSchema", "false")
    .option("multiLine", "true")
    .option("mode", "FAILFAST")
    .schema(orders_schema)
    .load(orders_path)
)

In [0]:
# Display schema

orders_df.printSchema()

# Show first five records

orders_df.show(5, truncate=False)

# Count records

orders_count = orders_df.count()

print("Orders row count:", orders_count)

In [0]:
reviews_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]

reviews_schema = create_string_schema(reviews_columns)

reviews_path = (
    RAW_PATH + "olist_order_reviews_dataset.csv"
)

reviews_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("enforceSchema", "false")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("mode", "FAILFAST")
    .schema(reviews_schema)
    .load(reviews_path)
)

reviews_df.printSchema()

reviews_df.show(5, truncate=100)

reviews_count = reviews_df.count()

print("Reviews row count:", reviews_count)

In [0]:
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

from datetime import datetime

In [0]:
# ==========================================
# PROJECT CONFIGURATION
# ==========================================

CATALOG = "olist_mlops"
SCHEMA = "bronze"

RAW_PATH = "/Volumes/olist_mlops/bronze/raw_files/"

SOURCE_SYSTEM = "kaggle_olist"

# Unique ID for this complete ingestion run
BATCH_ID = str(uuid4())

# Expected source row counts from our local profiling
DATASETS = {

    "customers": {
        "file": "olist_customers_dataset.csv",
        "rows": 99441,
        "multiline": False,
        "columns": [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    },

    "orders": {
        "file": "olist_orders_dataset.csv",
        "rows": 99441,
        "multiline": True,
        "columns": [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    },

    "order_items": {
        "file": "olist_order_items_dataset.csv",
        "rows": 112650,
        "multiline": False,
        "columns": [
            "order_id",
            "order_item_id",
            "product_id",
            "seller_id",
            "shipping_limit_date",
            "price",
            "freight_value"
        ]
    },

    "payments": {
        "file": "olist_order_payments_dataset.csv",
        "rows": 103886,
        "multiline": False,
        "columns": [
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value"
        ]
    },

    "reviews": {
        "file": "olist_order_reviews_dataset.csv",
        "rows": 99224,
        "multiline": True,
        "columns": [
            "review_id",
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    },

    "products": {
        "file": "olist_products_dataset.csv",
        "rows": 32951,
        "multiline": False,
        "columns": [
            "product_id",
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    },

    "sellers": {
        "file": "olist_sellers_dataset.csv",
        "rows": 3095,
        "multiline": False,
        "columns": [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    },

    "geolocation": {
        "file": "olist_geolocation_dataset.csv",
        "rows": 1000163,
        "multiline": False,
        "columns": [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "geolocation_city",
            "geolocation_state"
        ]
    },

    "category_translation": {
        "file": "product_category_name_translation.csv",
        "rows": 71,
        "multiline": False,
        "columns": [
            "product_category_name",
            "product_category_name_english"
        ]
    }
}

print("Configured datasets:", len(DATASETS))
print("Batch ID:", BATCH_ID)

In [0]:
def create_string_schema(columns):

    return StructType([
        StructField(
            column,
            StringType(),
            True
        )
        for column in columns
    ])

In [0]:
def ingest_dataset(table_name, config):

    print("\n" + "=" * 65)
    print(f"INGESTING DATASET: {table_name.upper()}")
    print("=" * 65)

    # --------------------------------------
    # 1. Identify source and target
    # --------------------------------------

    file_name = config["file"]

    source_path = RAW_PATH + file_name

    target_table = f"{CATALOG}.{SCHEMA}.{table_name}"

    expected_rows = config["rows"]

    columns = config["columns"]

    print("Source:", source_path)
    print("Target:", target_table)

    # --------------------------------------
    # 2. Construct explicit schema
    # --------------------------------------

    schema = create_string_schema(columns)

    # --------------------------------------
    # 3. Read source CSV
    # --------------------------------------

    source_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("enforceSchema", "false")
        .option("multiLine", str(config["multiline"]).lower())
        .option("escape", '"')
        .option("mode", "FAILFAST")
        .schema(schema)
        .load(source_path)
    )

    # --------------------------------------
    # 4. Validate source schema
    # --------------------------------------

    if source_df.columns != columns:

        raise ValueError(
            f"Column mismatch in {file_name}\n"
            f"Expected: {columns}\n"
            f"Received: {source_df.columns}"
        )

    # --------------------------------------
    # 5. Validate source row count
    # --------------------------------------

    source_count = source_df.count()

    if source_count != expected_rows:

        raise ValueError(
            f"Source count mismatch for {table_name}. "
            f"Expected {expected_rows}, "
            f"received {source_count}"
        )

    print("Source rows validated:", source_count)

    # --------------------------------------
    # 6. Add ingestion metadata
    # --------------------------------------

    bronze_df = (
        source_df

        .withColumn(
            "source_file",
            F.lit(file_name)
        )

        .withColumn(
            "source_system",
            F.lit(SOURCE_SYSTEM)
        )

        .withColumn(
            "ingestion_batch_id",
            F.lit(BATCH_ID)
        )

        .withColumn(
            "ingested_at",
            F.current_timestamp()
        )
    )

    # --------------------------------------
    # 7. Write managed Delta table
    # --------------------------------------

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    # --------------------------------------
    # 8. Validate Bronze table
    # --------------------------------------

    bronze_count = (
        spark.table(target_table).count()
    )

    if bronze_count != source_count:

        raise ValueError(
            f"Bronze validation failed for {table_name}. "
            f"Source={source_count}, "
            f"Bronze={bronze_count}"
        )

    # --------------------------------------
    # 9. Return ingestion metrics
    # --------------------------------------

    print("Bronze rows:", bronze_count)
    print("Status: SUCCESS")

    return {
        "table_name": target_table,
        "source_file": file_name,
        "source_rows": source_count,
        "bronze_rows": bronze_count,
        "status": "SUCCESS",
        "batch_id": BATCH_ID
    }

In [0]:
ingestion_results = []

for table_name, config in DATASETS.items():

    result = ingest_dataset(
        table_name,
        config
    )

    ingestion_results.append(result)

print("\nALL DATASETS PROCESSED")

In [0]:
total_source_rows = sum(
    result["source_rows"]
    for result in ingestion_results
)

total_bronze_rows = sum(
    result["bronze_rows"]
    for result in ingestion_results
)

print("Total source rows:", total_source_rows)
print("Total Bronze rows:", total_bronze_rows)

assert total_source_rows == total_bronze_rows

print("OVERALL INGESTION VALIDATION PASSED")

In [0]:
%sql
SHOW TABLES IN olist_mlops.bronze;

In [0]:
%sql

SELECT 'customers' AS table_name, COUNT(*) AS total_rows
FROM olist_mlops.bronze.customers

UNION ALL

SELECT 'orders', COUNT(*)
FROM olist_mlops.bronze.orders

UNION ALL

SELECT 'order_items', COUNT(*)
FROM olist_mlops.bronze.order_items

UNION ALL

SELECT 'payments', COUNT(*)
FROM olist_mlops.bronze.payments

UNION ALL

SELECT 'reviews', COUNT(*)
FROM olist_mlops.bronze.reviews

UNION ALL

SELECT 'products', COUNT(*)
FROM olist_mlops.bronze.products

UNION ALL

SELECT 'sellers', COUNT(*)
FROM olist_mlops.bronze.sellers

UNION ALL

SELECT 'geolocation', COUNT(*)
FROM olist_mlops.bronze.geolocation

UNION ALL

SELECT 'category_translation', COUNT(*)
FROM olist_mlops.bronze.category_translation;

In [0]:
%sql

SELECT
    order_id,
    order_status,
    source_file,
    source_system,
    ingestion_batch_id,
    ingested_at
FROM olist_mlops.bronze.orders
LIMIT 10;

In [0]:
%sql

DESCRIBE DETAIL olist_mlops.bronze.orders;

In [0]:
%sql

DESCRIBE HISTORY olist_mlops.bronze.orders;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS olist_mlops.mlops.ingestion_audit (
    ingestion_batch_id STRING,
    table_name STRING,
    source_file STRING,
    expected_rows BIGINT,
    source_rows BIGINT,
    bronze_rows BIGINT,
    delta_version BIGINT,
    status STRING,
    started_at TIMESTAMP,
    finished_at TIMESTAMP,
    error_message STRING
)
USING DELTA
COMMENT 'Persistent audit log for Olist Bronze ingestion';

In [0]:
%sql

DESCRIBE TABLE olist_mlops.mlops.ingestion_audit;

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)

AUDIT_TABLE = "olist_mlops.mlops.ingestion_audit"

AUDIT_SCHEMA = StructType([
    StructField("ingestion_batch_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("expected_rows", LongType(), True),
    StructField("source_rows", LongType(), True),
    StructField("bronze_rows", LongType(), True),
    StructField("delta_version", LongType(), True),
    StructField("status", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("finished_at", TimestampType(), True),
    StructField("error_message", StringType(), True)
])

In [0]:
def write_audit_record(
    table_name,
    config,
    status,
    started_at,
    result=None,
    delta_version=None,
    error_message=None
):

    record = {
        "ingestion_batch_id": BATCH_ID,
        "table_name": f"{CATALOG}.{SCHEMA}.{table_name}",
        "source_file": config["file"],
        "expected_rows": int(config["rows"]),
        "source_rows": (
            int(result["source_rows"])
            if result is not None else None
        ),
        "bronze_rows": (
            int(result["bronze_rows"])
            if result is not None else None
        ),
        "delta_version": delta_version,
        "status": status,
        "started_at": started_at,
        "finished_at": datetime.now(timezone.utc),
        "error_message": error_message
    }

    audit_df = spark.createDataFrame(
        [record],
        schema=AUDIT_SCHEMA
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT_TABLE)
    )

    print(f"Audit recorded: {table_name} -> {status}")

In [0]:
def get_latest_delta_version(table_name):

    history_df = spark.sql(
        f"DESCRIBE HISTORY {table_name} LIMIT 1"
    )

    latest_version = history_df.first()["version"]

    return int(latest_version)

In [0]:
def ingest_with_audit(table_name, config):

    started_at = datetime.now(timezone.utc)

    target_table = f"{CATALOG}.{SCHEMA}.{table_name}"

    try:

        # Execute our existing ingestion function.
        result = ingest_dataset(
            table_name,
            config
        )

        # Obtain the latest committed Delta version.
        delta_version = get_latest_delta_version(
            target_table
        )

    except Exception as error:

        error_message = (
            f"{type(error).__name__}: {str(error)}"
        )[:2000]

        print(f"\nINGESTION FAILED: {table_name}")
        print(error_message)

        # Attempt to preserve failure information.
        try:
            write_audit_record(
                table_name=table_name,
                config=config,
                status="FAILED",
                started_at=started_at,
                error_message=error_message
            )

        except Exception as audit_error:
            print(
                "WARNING: Failure audit could not be saved:",
                audit_error
            )

        # Propagate the original error.
        raise

    # Record success only after all checks pass.
    write_audit_record(
        table_name=table_name,
        config=config,
        status="SUCCESS",
        started_at=started_at,
        result=result,
        delta_version=delta_version
    )

    return result

In [0]:
# Generate a fresh batch ID for this execution.
BATCH_ID = str(uuid4())

print("Starting ingestion batch:", BATCH_ID)

run_results = []

for table_name, config in DATASETS.items():

    result = ingest_with_audit(
        table_name,
        config
    )

    run_results.append(result)

print("\n" + "=" * 60)
print("AUDITABLE BRONZE INGESTION COMPLETED")
print("=" * 60)

print("Batch ID:", BATCH_ID)
print("Tables processed:", len(run_results))

assert len(run_results) == len(DATASETS)

In [0]:
audit_results = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("ingestion_batch_id") == BATCH_ID)
    .select(
        "table_name",
        "expected_rows",
        "source_rows",
        "bronze_rows",
        "delta_version",
        "status"
    )
    .orderBy("table_name")
)

display(audit_results)

assert audit_results.count() == 9

assert (
    audit_results.filter(F.col("status") != "SUCCESS").count()
    == 0
)

print("All nine ingestion audit records validated.")

In [0]:
# Create a modified copy of the configuration.
test_config = {
    **DATASETS["category_translation"],
    "rows": 72
}

# Use a new ID for the simulated run.
BATCH_ID = str(uuid4())

print("Failure-test batch:", BATCH_ID)

try:

    ingest_with_audit(
        "category_translation",
        test_config
    )

except ValueError as error:

    print("\nExpected validation failure captured.")
    print("Reason:", error)

In [0]:
display(
    spark.table(AUDIT_TABLE)
    .filter(F.col("ingestion_batch_id") == BATCH_ID)
    .select(
        "table_name",
        "expected_rows",
        "source_rows",
        "bronze_rows",
        "status",
        "error_message"
    )
)

In [0]:
table_name = "olist_mlops.bronze.category_translation"

before_count = spark.table(table_name).count()

before_version = get_latest_delta_version(
    table_name
)

print("Before rerun:")
print("Rows:", before_count)
print("Version:", before_version)


# Run the dataset again with its correct configuration.
BATCH_ID = str(uuid4())

ingest_with_audit(
    "category_translation",
    DATASETS["category_translation"]
)


after_count = spark.table(table_name).count()

after_version = get_latest_delta_version(
    table_name
)

print("\nAfter rerun:")
print("Rows:", after_count)
print("Version:", after_version)


assert before_count == after_count == 71

assert after_version > before_version

print("\nRerun validation passed.")

In [0]:
%sql

DESCRIBE HISTORY
olist_mlops.bronze.category_translation;

In [0]:
previous_df = spark.sql(f"""
    SELECT
        ingestion_batch_id,
        COUNT(*) AS row_count
    FROM {table_name} VERSION AS OF {before_version}
    GROUP BY ingestion_batch_id
""")

current_df = spark.sql(f"""
    SELECT
        ingestion_batch_id,
        COUNT(*) AS row_count
    FROM {table_name} VERSION AS OF {after_version}
    GROUP BY ingestion_batch_id
""")

print("PREVIOUS VERSION:")
display(previous_df)

print("CURRENT VERSION:")
display(current_df)